### Hyper-tuning parameters of models (Logistic Regression, Random Forest, XGBoost)

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import make_scorer, fbeta_score, roc_auc_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [4]:
df = pd.read_csv("../../data/feature_extraction/er_blocking_candidates_k40_features_labeled.csv")

feature_cols = [
    "edit_ratio", "jaro_winkler", "lcs_ratio",
    "token_jaccard", "token_cosine",
    "tfidf_word_cosine", "tfidf_char_cosine",
    "dmetaphone_match",
]

X = df[feature_cols].astype(float).fillna(0.0)
y = df["label"].astype(int)

In [5]:
# As we decided to prioritise recall, we will use fbeta scorer
fbeta = make_scorer(fbeta_score, beta=2)

#### 1. Logistic Regression Grid

In [6]:
logreg = LogisticRegression(solver="liblinear", class_weight="balanced", max_iter=2000)

param_grid_lr = {
    "clf__C": [0.01, 0.1, 1, 10],
    "clf__penalty": ["l1", "l2"]
}

#### 2. Random Forest Grid

In [7]:
rf = RandomForestClassifier(class_weight="balanced", random_state=42)

param_grid_rf = {
    "clf__n_estimators": [200, 400, 800],
    "clf__max_depth": [None, 10, 20],
    "clf__min_samples_split": [2, 5, 10],
}

#### 3. XGBoost Grid

In [8]:
xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=y.value_counts()[0] / y.value_counts()[1],
    random_state=42,
    n_jobs=-1
)

param_grid_xgb = {
    "clf__n_estimators": [300, 600, 900],
    "clf__max_depth": [4, 6, 8],
    "clf__learning_rate": [0.01, 0.05, 0.1],
    "clf__subsample": [0.8, 0.9, 1.0],
    "clf__colsample_bytree": [0.8, 0.9, 1.0],
}

In [9]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [16]:
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", logreg)
])

grid_lr = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid_lr,
    scoring=fbeta,         
    cv=cv,
    n_jobs=-1,
    verbose=2
)

grid_lr.fit(X, y)
print("Best LR params:", grid_lr.best_params_)
print("Best F2-score:", grid_lr.best_score_)

Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best LR params: {'clf__C': 0.01, 'clf__penalty': 'l1'}
Best F2-score: 0.8074427364792737


In [ ]:
pipe_rf = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", rf)
])

grid_rf = GridSearchCV(
    estimator=pipe_rf,
    param_grid=param_grid_rf,
    scoring=fbeta,          
    cv=cv,
    n_jobs=-1,
    verbose=2
)

grid_rf.fit(X, y)
print("Best RF params:", grid_rf.best_params_)
print("Best F2-score:", grid_rf.best_score_)

Fitting 5 folds for each of 27 candidates, totalling 135 fits


In [ ]:
pipe_xgb = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", xgb)
])

grid_xgb = GridSearchCV(
    estimator=pipe_xgb,
    param_grid=param_grid_xgb,
    scoring=fbeta,         
    cv=cv,
    n_jobs=-1,
    verbose=2
)

grid_xgb.fit(X, y)
print("Best XGB params:", grid_xgb.best_params_)
print("Best F2-score:", grid_xgb.best_score_)

In [ ]:
# Summarizing results:
import pandas as pd

results = pd.DataFrame([
    {"Model": "Logistic Regression", "Best Score (F2)": grid_lr.best_score_, "Params": grid_lr.best_params_},
    {"Model": "Random Forest", "Best Score (F2)": grid_rf.best_score_, "Params": grid_rf.best_params_},
    {"Model": "XGBoost", "Best Score (F2)": grid_xgb.best_score_, "Params": grid_xgb.best_params_},
])

display(results)